In [14]:
import os
from google.colab import userdata
from chromadb.utils import embedding_functions

# 1. 코랩의 Secret 설정에서 'OPENAI_API_KEY'를 가져옵니다.
try:
    api_key = userdata.get('OPENAI_API_KEY')
except:
    # 만약 설정되어 있지 않다면 직접 입력받거나 환경변수를 확인
    api_key = os.getenv("OPENAI_API_KEY")

# 2. 값이 제대로 들어왔는지 확인 (None이면 에러 발생)
if not api_key:
    raise ValueError("API Key가 설정되지 않았습니다. 코랩 왼쪽 사이드바의 '열쇠 모양(Secrets)' 아이콘에서 'OPENAI_API_KEY'를 추가해주세요.")

# 3. ChromaDB가 요구하는 환경변수명으로 할당 (반드시 문자열로 변환)
os.environ["CHROMA_OPENAI_API_KEY"] = str(api_key)

# 4. 임베딩 함수 생성
openai_ef = embedding_functions.OpenAIEmbeddingFunction(
    api_key=str(api_key),
    model_name="text-embedding-3-small"
)

print("✅ API Key가 성공적으로 설정되었습니다.")

✅ API Key가 성공적으로 설정되었습니다.


### RAG (Review analysis only)


In [16]:

def query_vector_db(collection, query_text):
    print(f"\n=== Step 2: RAG 검색 쿼리 실행 ===")
    print(f"사용자 질문: '{query_text}'")

    # 상위 3개 검색
    results = collection.query(
        query_texts=[query_text],
        n_results=3
    )

    result_md = f"# RAG 검색 결과\n\n**질문:** {query_text}\n\n"

    # 결과 출력 포맷팅 및 마크다운 구성
    print("\n[검색된 의미 있는 상위 3개 식당 데이터]")
    for i in range(len(results['ids'][0])):
        res_id = results['ids'][0][i]
        meta = results['metadatas'][0][i]
        dist = results['distances'][0][i]
        doc = results['documents'][0][i]

        print(f"\n--- Rank {i+1} ---")
        print(f"식당 이름: {meta['restaurant_name']} (거리: {dist:.4f})")

        # 전체 문서를 다 출력하면 기니, 앞부분 200자만 출력
        preview = doc[:300] + "..." if len(doc) > 300 else doc
        print(f"데이터 요약: {preview}")

        result_md += f"## Rank {i+1} : {meta['restaurant_name']}\n"
        result_md += f"- **유사도 거리(Distance):** {dist:.4f}\n"
        result_md += f"- **데이터 전체 요약:** {doc}\n\n"

    md_path = "./vectorDB/rag_query_result.md"
    with open(md_path, "w", encoding="utf-8") as f:
        f.write(result_md)
    print(f"\n✅ 검색 결과가 '{md_path}' 파일로 저장되었습니다.")


In [19]:
if __name__ == "__main__":
    col = build_vector_db()
    query_vector_db(col, "강남역 단체 회식 삼겹살집 추천해줘")

=== Step 1: Vector DB 구축 시작 ===
총 2838개의 리뷰 데이터를 로드했습니다.
식당 단위로 취합된 개수: 64개
64 / 64 개 처리 완료...
=== Step 1: Vector DB 구축 완료 ===

=== Step 2: RAG 검색 쿼리 실행 ===
사용자 질문: '강남역 단체 회식 삼겹살집 추천해줘'

[검색된 의미 있는 상위 3개 식당 데이터]

--- Rank 1 ---
식당 이름: 대통령삼겹살 신논현역점 (거리: 0.4671)
데이터 요약: 식당 이름: 대통령삼겹살 신논현역점
주요 강점: 직원들이 친절하다, 매장이 깔끔하다, 분위기도 좋아요, 고기가 너무 맛있어요, 편하게 잘 먹을 수 있었음, 고기가 정말 차원이 다르다, 고기 질이 좋음, 매장 분위기가 편하다, 훈연 향이 잘 입혀져 있다, 단체 모임에 적합한 테이블이 많음, 특별한 모임에 적합, 고기를 구워주는 서비스가 편리하다, 맛있음, 맛있고 분위기가 좋음, 좋은 매장 분위기, 대나무 초벌 삼겹살이 독특하고 맛있음, 고기 맛이 좋음, 고기 숯향이 좋음, 친절하다, 서비스, 친절한 서비스, 맛있는 음식, 구워주는 서...

--- Rank 2 ---
식당 이름: 돈육지존 역삼역본점 (거리: 0.4809)
데이터 요약: 식당 이름: 돈육지존 역삼역본점
주요 강점: 반찬이 다 맛있다, 매장이 깔끔하다, 만족도가 높다, 맛있는 고기, 친절하게 안내해주시고, 뼈삼겹살이 부드럽고 고소하다, 삼겹살 맛, 직원 분들 친절함, 고기 질이 좋음, 시골청국장이 맛있다, 맛있음, 부드러운 뼈삼겹, 청국장 맛집, 맛있게 먹었다, 모임 분위기가 좋다, 연기가 차지 않음, 삼겹살 맛있음, 주차 공간이 있어 편리하다, 고기 맛이 좋음, 구워주는 서비스, 친절한 서비스, 단골이 될 예정, 사이드 메뉴 구성 좋음, 최고의 고깃집, 뼈삼겹살이 맛있음, 고기가 정말 좋다, 직접 ...

--- Rank 3 ---
식당 이름: 강남 돼지상회 무한리필 (거리: 0.4970)
데이터 요약: 식당 이름: 강남 

### RAG (3 Files)

In [26]:
import os
import json
import chromadb
from chromadb.utils import embedding_functions
from google.colab import userdata

# 환경 변수 설정
os.environ["CHROMA_OPENAI_API_KEY"] = str(userdata.get('OPENAI_API_KEY'))

def build_integrated_vector_db():
    print("=== Step 1: 3개 파일 데이터 통합 및 DB 구축 ===")

    # 1. 파일 로드
    with open("./vectorDB/final_restaurants.json", "r", encoding="utf-8") as f: rest_base = json.load(f)
    with open("./vectorDB/final_reviews.json", "r", encoding="utf-8") as f: all_reviews = json.load(f)
    with open("./vectorDB/restaurant_analysis_2838_2.json", "r", encoding="utf-8") as f: rest_analysis = json.load(f)

    # 2. 데이터 통합 (ID 기준 매핑)
    # final_restaurants.json의 키는 "ID"
    master_data = {str(r['ID']): r for r in rest_base}

    # 리뷰 병합 (R_ID 기준)
    for review in all_reviews:
        rid = str(review.get("R_ID"))
        if rid in master_data:
            if "reviews" not in master_data[rid]: master_data[rid]["reviews"] = []
            master_data[rid]["reviews"].append(review.get("detail", ""))

    # 분석 데이터 병합 (restaurant_id 기준)
    for analysis in rest_analysis:
        rid = str(analysis.get("restaurant_id"))
        if rid in master_data:
            if "analysis" not in master_data[rid]: master_data[rid]["analysis"] = []
            master_data[rid]["analysis"].append(analysis)

    # 3. ChromaDB 설정
    client = chromadb.PersistentClient(path="./integrated_db")
    openai_ef = embedding_functions.OpenAIEmbeddingFunction(
        api_key=os.environ["CHROMA_OPENAI_API_KEY"],
        model_name="text-embedding-3-small"
    )
    collection = client.get_or_create_collection(name="full_restaurant_data", embedding_function=openai_ef)

    # 4. Document 생성
    documents, metadatas, ids = [], [], []
    for rid, data in master_data.items():
        # 데이터 타입 체크: 딕셔너리가 아니면 빈 딕셔너리로 취급
        add_info = data.get('additional_info')
        if not isinstance(add_info, dict):
            add_info = {}

        # 상세 텍스트 구성 (검색용)
        info_str = f"식당명: {data.get('name', 'N/A')}\n"
        info_str += f"주소: {data.get('address', '정보없음')}\n"

        # 수정된 부분: 안전하게 add_info 사용
        info_str += f"주차 및 찾아오는 길: {add_info.get('찾아오는길_및_주차안내', '정보없음')}\n"
        info_str += f"편의시설: {', '.join(add_info.get('편의시설', []))}\n"

        reviews = data.get("reviews", [])
        info_str += f"고객 리뷰: {' | '.join(reviews[:10])}"

        documents.append(info_str)
        metadatas.append({
            "name": data.get('name', ''),
            "restaurant_id": rid,
            "address": data.get('address', '')
        })
        ids.append(rid)

    # 5. DB 삽입
    collection.add(documents=documents, metadatas=metadatas, ids=ids)
    print("=== Step 1: 통합 DB 구축 완료 ===")
    return collection
다.
    # 예: "주차 가능한 영동그집 찾아가는 방법 알려줘"

=== Step 1: 3개 파일 데이터 통합 및 DB 구축 ===
=== Step 1: 통합 DB 구축 완료 ===


In [29]:
if __name__ == "__main__":
    col = build_vector_db()
    query_vector_db(col, "주차가능하고 서비스 좋은 삼겹살집 추천해줘")

=== Step 1: Vector DB 구축 시작 ===
이미 DB에 64개의 데이터가 존재합니다.

=== Step 2: RAG 검색 실행 ===

[Rank 1] 육성급 삼성본점
메타데이터: {'restaurant_id': '1275553707', 'restaurant_name': '육성급 삼성본점'}
요약: 식당 이름: 육성급 삼성본점
주요 강점: 직원분들이 엄청 친절함, 직원들이 친절하다, 좋은 메뉴 구성, 고기를 직접 구워주는 서비스, 좋은 분위기, 고기 맛있음, 사장님이 친절하시...

[Rank 2] 육쌈마을 논현역본점
메타데이터: {'restaurant_name': '육쌈마을 논현역본점', 'restaurant_id': '2060948347'}
요약: 식당 이름: 육쌈마을 논현역본점
주요 강점: 직원들이 친절하다, 매장이 깔끔하다, 맛깔난 우렁된장찌개, 고기 맛있어요, 고기 맛있음, 우렁된장찌개가 완벽하다, 우렁된장찌개가 맛있었...

[Rank 3] 봉우화로
메타데이터: {'restaurant_id': '11609540', 'restaurant_name': '봉우화로'}
요약: 식당 이름: 봉우화로
주요 강점: 매장이 넓고 청결하다, 양이 많다, 반찬도 잘 나옴, 직원들이 친절하다, 이베리코 목살이 맛있다, 우삼겹과 갈비살의 가성비가 좋음, 맛있고 친절함...
